# Traffic Demand Prediction — Full Solution

**Goal:** predict `demand` for ~41.8k test rows. Metric: `max(0, 100·R²)`.

**Sections:** EDA → Preprocessing → Baseline → Improvements → Final Model → Submission

### Key insights driving the approach
- **Regression** on a smooth spatiotemporal demand surface.
- Train = day 48 (full 24h) + day 49 (00:00–02:00); **Test = day 49, 02:15–13:45**.
- `geohash` strings **decode to (lat, lon)**; 99.9% of test geohashes appear in train.
- A faithful CV proxy is **full-train random 5-fold** — it reproduces the prior leaderboard score (0.9125) almost exactly, because test points are interpolatable from the day-48 surface at the same geohash/time-of-day.
- The single biggest lever is **out-of-fold geohash target encoding** (per-geohash demand level), which lifts CV from ~0.92 to ~0.95.

*Hardware:* MacBook M1 Pro — tree models use `n_jobs=-1`; the neural net uses the MPS backend; all features stored as `float32`.


## 0. Imports & Configuration

In [ ]:
import os, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge

RANDOM_STATE = 42
N_FOLDS = 5
DATA_DIR = "dataset"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
np.random.seed(RANDOM_STATE)
print("Libraries loaded. LightGBM", lgb.__version__, "| XGBoost", xgb.__version__)


## 1. Exploratory Data Analysis (EDA)

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print("train:", train.shape, "| test:", test.shape, "| sample_submission:", sample_sub.shape)
print("\nTrain dtypes:\n", train.dtypes)
train.head()


In [ ]:
# --- Target distribution ---
print("Target `demand` statistics:")
print(train["demand"].describe())
print("min/max:", train.demand.min(), "/", train.demand.max(), "| negatives:", (train.demand < 0).sum())

# --- Missing values ---
print("\nMissing values (train):")
print(train.isna().sum()[lambda s: s > 0])
print("\nMissing values (test):")
print(test.isna().sum()[lambda s: s > 0])

# --- Categorical cardinality ---
print("\nCardinality / value counts:")
for c in ["geohash", "RoadType", "NumberofLanes", "LargeVehicles", "Landmarks", "Weather"]:
    print(f"  {c}: {train[c].nunique()} unique")


In [ ]:
# --- Temporal structure: this defines the whole strategy ---
def ts_to_min(s):
    h, m = str(s).split(":")
    return int(h) * 60 + int(m)

for df in (train, test):
    df["tmin"] = df["timestamp"].map(ts_to_min)

print("Train day / time coverage:")
for d, g in train.groupby("day"):
    print(f"  day {d}: {len(g):6d} rows | tmin {g.tmin.min():>4}-{g.tmin.max():<4} | {g.tmin.nunique()} slots")
print("Test  day / time coverage:")
for d, g in test.groupby("day"):
    print(f"  day {d}: {len(g):6d} rows | tmin {g.tmin.min():>4}-{g.tmin.max():<4} | {g.tmin.nunique()} slots")

# Geohash overlap between train and test
overlap = test.geohash.isin(set(train.geohash)).mean()
print(f"\nFraction of test geohashes present in train: {overlap:.4f}")
print("=> Test = future window of day 49; geohashes are known from day 48's full-day surface.")


## 2. Preprocessing & Feature Engineering

Deterministic, leakage-safe transforms applied identically to train/test:
- **geohash → (lat, lon)** via base-32 geohash decoding (gives a continuous spatial map).
- **timestamp → minutes-of-day** plus cyclic harmonics (sin/cos at 1×, 2×, 3× daily frequency).
- **Missing-value handling:** numeric → median; categoricals → an explicit `"Missing"` level (+ missing-flag for Temperature).

Target-dependent features (geohash target encoding & per-geohash demand stats) are **not** computed here — they are built *inside each CV fold* (Section 4) to prevent leakage.

In [ ]:
# --- Geohash decoder: base-32 string -> (lat, lon) center ---
_BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"

def decode_geohash(gh):
    lat_lo, lat_hi = -90.0, 90.0
    lon_lo, lon_hi = -180.0, 180.0
    even = True
    for ch in gh:
        cd = _BASE32.index(ch)
        for mask in (16, 8, 4, 2, 1):
            if even:
                mid = (lon_lo + lon_hi) / 2
                if cd & mask: lon_lo = mid
                else:         lon_hi = mid
            else:
                mid = (lat_lo + lat_hi) / 2
                if cd & mask: lat_lo = mid
                else:         lat_hi = mid
            even = not even
    return (lat_lo + lat_hi) / 2, (lon_lo + lon_hi) / 2

# Decode every geohash once (cached lookup)
_all_gh = pd.concat([train.geohash, test.geohash]).unique()
GH2LATLON = {g: decode_geohash(g) for g in _all_gh}

CAT_COLS = ["RoadType", "LargeVehicles", "Landmarks", "Weather"]

def build_static_features(df):
    """Leakage-free, deterministic features (no target involved)."""
    d = df.copy()
    d["lat"] = d.geohash.map(lambda g: GH2LATLON[g][0]).astype("float32")
    d["lon"] = d.geohash.map(lambda g: GH2LATLON[g][1]).astype("float32")
    d["hour"] = (d.tmin // 60).astype("float32")
    # cyclic time-of-day harmonics
    for k in (1, 2, 3):
        d[f"sin{k}"] = np.sin(2 * np.pi * k * d.tmin / 1440).astype("float32")
        d[f"cos{k}"] = np.cos(2 * np.pi * k * d.tmin / 1440).astype("float32")
    d["tmin"] = d["tmin"].astype("float32")
    # Temperature: median impute + missing flag
    d["Temp_missing"] = d.Temperature.isna().astype("float32")
    d["Temperature"]  = d.Temperature.fillna(train.Temperature.median()).astype("float32")
    d["NumberofLanes"] = d.NumberofLanes.astype("float32")
    # categoricals: explicit "Missing" level
    for c in CAT_COLS:
        d[c] = d[c].fillna("Missing").astype("category")
    return d

train_fe = build_static_features(train)
test_fe  = build_static_features(test)
print("Static features built. lat range:",
      round(float(train_fe.lat.min()), 3), "to", round(float(train_fe.lat.max()), 3))

STATIC_FEATS = (["lat", "lon", "tmin", "hour", "NumberofLanes", "Temperature", "Temp_missing"]
                + [f"{p}{k}" for k in (1, 2, 3) for p in ("sin", "cos")]
                + CAT_COLS)
y = train_fe["demand"].values.astype("float32")
print("N static features:", len(STATIC_FEATS))


In [ ]:
# --- Leakage-safe target-encoding helpers (fit on TRAIN part of a fold only) ---
def smoothed_mean_encoding(frame, key_cols, target, smoothing=10.0):
    """Smoothed target mean per group, blended toward the global mean."""
    grp = frame.groupby(key_cols)[target]
    means, counts = grp.mean(), grp.count()
    glob = frame[target].mean()
    enc = (means * counts + glob * smoothing) / (counts + smoothing)
    return enc, glob

def add_target_features(tr_part, other_parts):
    """Build target-derived features from tr_part and map onto every frame in
    [tr_part] + other_parts. Returns the list of feature names added.
    `tr_part` must contain the '__y' target column; other parts need not."""
    added = []
    frames = [tr_part] + list(other_parts)

    # 1) geohash target encoding (overall demand level per location)
    enc, glob = smoothed_mean_encoding(tr_part, "geohash", "__y", smoothing=10.0)
    gstd = tr_part.groupby("geohash")["__y"].std()
    for f in frames:
        f["gh_te"]  = f.geohash.map(enc).fillna(glob).astype("float32")
        f["gh_std"] = f.geohash.map(gstd).fillna(0.0).astype("float32")
    added += ["gh_te", "gh_std"]

    # 2) geohash x hour encoding (time-of-day demand profile per location)
    enc2, glob2 = smoothed_mean_encoding(tr_part, ["geohash", "hour"], "__y", smoothing=5.0)
    for f in frames:
        idx = f.set_index(["geohash", "hour"]).index
        f["gh_hr_te"] = pd.Series(idx.map(enc2), index=f.index).astype("float32")
        f["gh_hr_te"] = f["gh_hr_te"].fillna(f["gh_te"])  # back off to geohash level
    added += ["gh_hr_te"]
    return added

print("Target-encoding helpers ready.")


## 3. Validation Strategy

The faithful proxy for the leaderboard is a **random 5-fold split over the full training set**. Empirically this reproduces the previously observed leaderboard score (0.9125) almost exactly, because every test row is a point on the same smooth (geohash, time-of-day) surface that day 48 densely samples — so random K-fold, not a day-holdout, matches the real generalization gap.

All target-derived features are recomputed **inside each fold** on the training partition only, so the out-of-fold predictions are leakage-free and the reported R² is trustworthy.